In [1]:
# Cell 0: setup — imports, repository discovery, and shared helpers
import os
import sys
import subprocess
import importlib.util
import time
from collections import Counter
from pathlib import Path
from urllib.parse import urlparse

def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "requirements.txt").is_file():
            return candidate
    raise RuntimeError(f"Could not find repository root above {current}")

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

try:
    import httpx
except ImportError:
    # Jupyter may select the base Anaconda kernel; reuse the existing project venv without installing.
    venv_sites = [
        REPO_ROOT / ".venv" / "Lib" / "site-packages",
        REPO_ROOT / ".venv" / "lib" / f"python{sys.version_info.major}.{sys.version_info.minor}" / "site-packages",
    ]
    for site_dir in venv_sites:
        if site_dir.is_dir() and str(site_dir) not in sys.path:
            sys.path.insert(0, str(site_dir))
    try:
        import httpx
    except ImportError as exc:
        raise RuntimeError(
            "httpx is required. Use the repository's Python 3.12 .venv; "
            "this notebook intentionally does not install packages."
        ) from exc

# Load only models.py; importing src.search would eagerly load the full pandas/LangGraph pipeline.
models_path = REPO_ROOT / "src" / "search" / "models.py"
models_spec = importlib.util.spec_from_file_location("_searxng_spike_models", models_path)
if models_spec is None or models_spec.loader is None:
    raise RuntimeError(f"Cannot load RawCandidate from {models_path}")
models_module = importlib.util.module_from_spec(models_spec)
sys.modules[models_spec.name] = models_module
models_spec.loader.exec_module(models_module)
RawCandidate = models_module.RawCandidate

SEARXNG_URL = os.environ.get("SEARXNG_URL", "http://localhost:8888").rstrip("/")
SX_LAST_ERROR: str | None = None

def sx_search(
    q: str,
    *,
    language: str | None = None,
    k: int = 10,
    engines: list[str] | tuple[str, ...] | None = None,
    timeout: float = 20,
) -> tuple[list[dict], dict]:
    """Run one synchronous SearXNG JSON search and return results plus diagnostics."""
    params: dict[str, str] = {"q": q, "format": "json"}
    if language:
        params["language"] = language
    if engines:
        params["engines"] = ",".join(engines)
    started = time.perf_counter()
    response = httpx.get(
        f"{SEARXNG_URL}/search", params=params, timeout=timeout, follow_redirects=True
    )
    elapsed = time.perf_counter() - started
    response.raise_for_status()
    data = response.json()
    if not isinstance(data, dict):
        raise ValueError(f"Expected a JSON object, got {type(data).__name__}")
    results = data.get("results") or []
    if not isinstance(results, list):
        raise ValueError("SearXNG response field 'results' is not a list")
    meta = {
        "elapsed_s": elapsed,
        "unresponsive_engines": data.get("unresponsive_engines") or [],
        "number_of_results": data.get("number_of_results"),
    }
    return results[:k], meta

def sx_up() -> bool:
    """Return True only when the endpoint serves JSON; retain a useful failure reason."""
    global SX_LAST_ERROR
    try:
        results, meta = sx_search("searxng healthcheck", k=1, timeout=5)
        ready = isinstance(results, list) and isinstance(meta, dict)
        SX_LAST_ERROR = None if ready else "response did not satisfy the JSON contract"
        return ready
    except Exception as exc:
        SX_LAST_ERROR = f"{type(exc).__name__}: {exc}"
        return False

def sx_unavailable_message() -> str:
    detail = SX_LAST_ERROR or "unknown error"
    return f"SearXNG 未就绪，跳过（{detail}）"

print(f"Repository root: {REPO_ROOT}")
print(f"SearXNG URL: {SEARXNG_URL}")
print(f"Instance ready: {sx_up()}")


Repository root: /Users/kumo/programming/competitor_product_search
SearXNG URL: http://localhost:8888
Instance ready: True


In [2]:
# Cell 1: public instance reproducibility check
# Public services change frequently; this cell rechecks the 17-instance baseline.
PUBLIC_INSTANCES = [
    "priv.au",
    "search.inetol.net",
    "opnxng.com",
    "search.rhscz.eu",
    "searx.tiekoetter.com",
    "paulgo.io",
    "searx.perennialte.ch",
    "searx.be",
    "baresearch.org",
    "search.hbubli.cc",
    "search.disroot.org",
    "searxng.site",
    "copp.gg",
    "northboot.xyz",
    "searxng.world",
    "search.bus-hit.me",
    "search.projectsegfau.lt",
]

print("host | HTTP code | JSON?")
print("-" * 72)
with httpx.Client(timeout=8, follow_redirects=True) as client:
    for host in PUBLIC_INSTANCES:
        status: str | int = "ERR"
        is_json = False
        try:
            response = client.get(
                f"https://{host}/search", params={"q": "test", "format": "json"}
            )
            status = response.status_code
            try:
                payload = response.json()
                is_json = isinstance(payload, dict) and isinstance(payload.get("results"), list)
            except (ValueError, TypeError):
                pass
        except httpx.HTTPError as exc:
            status = type(exc).__name__
        print(f"{host} | {status} | {'yes' if is_json else 'no'}")

print("\nBaseline from 2026-08-14: 0/17 exposed a usable JSON endpoint.")
print("Public-instance behavior is volatile; self-hosting is required for the remaining cells.")


host | HTTP code | JSON?
------------------------------------------------------------------------
priv.au | 429 | no
search.inetol.net | 429 | no
opnxng.com | 429 | no
search.rhscz.eu | 429 | no
searx.tiekoetter.com | 429 | no
paulgo.io | 429 | no
searx.perennialte.ch | 429 | no
searx.be | 200 | no
baresearch.org | 200 | no
search.hbubli.cc | 429 | no
search.disroot.org | 429 | no
searxng.site | 403 | no
copp.gg | 502 | no
northboot.xyz | ConnectError | no
searxng.world | ConnectError | no
search.bus-hit.me | ConnectError | no
search.projectsegfau.lt | 200 | no

Baseline from 2026-08-14: 0/17 exposed a usable JSON endpoint.
Public-instance behavior is volatile; self-hosting is required for the remaining cells.


In [3]:
# Cell 2: write config and idempotently start/reuse the disposable Docker container
config_dir = REPO_ROOT / "src" / "search" / "script" / ".searxng"
config_dir.mkdir(parents=True, exist_ok=True)
settings_path = config_dir / "settings.yml"
settings_path.write_text(
    '''use_default_settings: true
server:
  secret_key: "searxng-feasibility-spike"
  limiter: false
  public_instance: false
  image_proxy: false
search:
  formats: [html, json]
  safe_search: 0
''',
    encoding="utf-8",
)

print(f"Wrote: {settings_path}")
container_name = "searxng-spike"
docker_run = [
    "docker", "run", "--rm", "-d", "--name", container_name,
    "-p", "8888:8080", "-v", f"{config_dir}:/etc/searxng",
    "docker.io/searxng/searxng:latest",
]

def run_local(command: list[str], timeout: float = 30) -> subprocess.CompletedProcess[str]:
    return subprocess.run(command, text=True, capture_output=True, timeout=timeout, check=False)

def docker_ready() -> bool:
    try:
        return run_local(["docker", "info"], timeout=10).returncode == 0
    except (FileNotFoundError, subprocess.TimeoutExpired):
        return False

if not docker_ready() and sys.platform == "darwin":
    print("Docker daemon 未启动；正在打开 Docker Desktop…")
    run_local(["open", "-a", "Docker"], timeout=15)
    for _ in range(45):
        if docker_ready():
            break
        time.sleep(2)

if not docker_ready():
    print("ERROR: Docker daemon 不可用。请先启动 Docker Desktop，然后重新运行本 cell。")
else:
    state = run_local(["docker", "inspect", "-f", "{{.State.Running}}", container_name])
    if state.returncode == 0 and state.stdout.strip() == "true":
        print(f"Reusing running container: {container_name}")
    elif state.returncode == 0:
        started = run_local(["docker", "start", container_name], timeout=30)
        print(started.stdout.strip() or started.stderr.strip())
    else:
        started = run_local(docker_run, timeout=180)
        print(started.stdout.strip() or started.stderr.strip())

    for _ in range(30):
        if sx_up():
            break
        time.sleep(2)
    if sx_up():
        print(f"Instance ready: {SEARXNG_URL}")
        print(f"When finished: docker stop {container_name}")
    else:
        logs = run_local(["docker", "logs", "--tail", "40", container_name])
        print(f"ERROR: {sx_unavailable_message()}")
        print(logs.stdout[-4000:] or logs.stderr[-4000:])


Wrote: /Users/kumo/programming/competitor_product_search/src/search/script/.searxng/settings.yml
Reusing running container: searxng-spike
Instance ready: http://localhost:8888
When finished: docker stop searxng-spike


In [4]:
# Cell 3: basic feasibility and RawCandidate field contract
if not sx_up():
    print(sx_unavailable_message())
else:
    query = "Magic Rock Saucery 4 X 330ML Tesco"
    results, meta = sx_search(query, k=10)
    print(f"Query: {query!r}")
    print(f"Results: {len(results)} in {meta['elapsed_s']:.2f}s")
    print(f"Unresponsive engines: {meta['unresponsive_engines']}")
    print(f"Raw fields: {sorted(results[0]) if results else []}\n")

    for index, item in enumerate(results):
        print(f"[{index}] title: {item.get('title', 'N/A')[:120]}")
        print(f"    url:     {item.get('url', 'N/A')[:120]}")
        print(f"    content: {item.get('content', 'N/A')[:120]}")

    candidates = [
        RawCandidate(
            title=item["title"],
            url=item["url"],
            snippet=item.get("content") or "",
        )
        for item in results
        if item.get("title") and item.get("url")
    ]
    tesco_hits = [candidate for candidate in candidates if "tesco.com" in candidate.url.lower()]
    print(f"\nMapped to {len(candidates)} RawCandidate objects")
    print(f"Tesco URLs found: {len(tesco_hits)}")


Query: 'Magic Rock Saucery 4 X 330ML Tesco'
Results: 0 in 0.04s
Unresponsive engines: [['brave', 'Suspended: too many requests'], ['duckduckgo', 'Suspended: access denied'], ['google cse', 'Suspended: too many requests'], ['startpage', 'Suspended: CAPTCHA']]
Raw fields: []


Mapped to 0 RawCandidate objects
Tesco URLs found: 0


In [5]:
# Cell 4: format compatibility across the same five query groups as duckduckgo.ipynb
if not sx_up():
    print(sx_unavailable_message())
else:
    test_queries = [
        ("Heinz Baked Beans 415g Tesco", "grocery — Tesco"),
        ("Samsung Galaxy S24 Ultra Amazon", "electronics — Amazon"),
        ("Bosch Serie 4 Washing Machine Argos", "appliance — Argos"),
        ("Kopparberg Strawberry & Lime 500ml", "alcohol — any retailer"),
        ("LEGO Star Wars Millennium Falcon", "toys — any retailer"),
    ]
    all_ok = True
    for query, category in test_queries:
        try:
            results, meta = sx_search(query, k=5)
        except Exception as exc:
            print(f"ERROR {query!r}: {type(exc).__name__}: {exc}")
            all_ok = False
            continue
        missing_url = sum(not str(item.get("url") or "").strip() for item in results)
        missing_title = sum(not str(item.get("title") or "").strip() for item in results)
        domains = Counter(
            urlparse(str(item.get("url") or "")).netloc.lower()
            for item in results
            if item.get("url")
        )
        domains.pop("", None)
        all_ok = all_ok and missing_url == 0 and missing_title == 0
        print(
            f"[{category}] {len(results)} results; missing url={missing_url}, "
            f"missing title={missing_title}; {meta['elapsed_s']:.2f}s"
        )
        print(f"    netlocs: {dict(domains)}")
    print(f"\nAll returned rows satisfy the search-layer minimum fields: {all_ok}")


[grocery — Tesco] 0 results; missing url=0, missing title=0; 0.05s
    netlocs: {}
[electronics — Amazon] 0 results; missing url=0, missing title=0; 0.04s
    netlocs: {}
[appliance — Argos] 0 results; missing url=0, missing title=0; 0.05s
    netlocs: {}
[alcohol — any retailer] 0 results; missing url=0, missing title=0; 0.05s
    netlocs: {}
[toys — any retailer] 0 results; missing url=0, missing title=0; 0.04s
    netlocs: {}

All returned rows satisfy the search-layer minimum fields: True


In [6]:
# Cell 5: country targeting — the key feasibility test
if not sx_up():
    print(sx_unavailable_message())
else:
    _COUNTRY_TO_LANGUAGE = {
        "uk": "en-GB", "gb": "en-GB", "de": "de-DE", "fr": "fr-FR",
        "us": "en-US", "nl": "nl-NL", "jp": "ja-JP", "es": "es-ES",
        "it": "it-IT", "pt": "pt-PT", "se": "sv-SE", "pl": "pl-PL",
        "br": "pt-BR", "au": "en-AU", "ca": "en-CA",
    }
    local_cases = {
        "uk": ("Nescafe Gold Tesco", "tesco.com"),
        "de": ("Nescafe Gold Amazon Deutschland", "amazon.de"),
        "fr": ("Nescafe Gold Amazon France", "amazon.fr"),
        "us": ("Nescafe Gold Amazon USA", "amazon.com"),
    }

    def result_profile(results: list[dict]) -> tuple[Counter, list[str]]:
        hosts = sorted({
            urlparse(str(item.get("url") or "")).netloc.lower().removeprefix("www.")
            for item in results if item.get("url")
        } - {""})
        return Counter(host.rsplit(".", 1)[-1] for host in hosts), hosts

    local_target_hits: dict[str, int] = {}
    for country, (query, target_domain) in local_cases.items():
        results, meta = sx_search(query, language=_COUNTRY_TO_LANGUAGE[country], k=10)
        tlds, hosts = result_profile(results)
        hits = sum(target_domain in urlparse(item.get("url", "")).netloc.lower() for item in results)
        local_target_hits[country] = hits
        print(f"{country} / {_COUNTRY_TO_LANGUAGE[country]}: {len(results)} results, target hits={hits}")
        print(f"    TLDs: {dict(tlds)}")
        print(f"    domains: {hosts}")

    control_query = "Nescafe Gold Amazon"
    control_targets = {"uk": "amazon.co.uk", "de": "amazon.de", "fr": "amazon.fr", "us": "amazon.com"}
    controls: dict[str, set[str]] = {}
    for label, language in [("none", None), *[(code, _COUNTRY_TO_LANGUAGE[code]) for code in ("uk", "de", "fr", "us")]]:
        results, _ = sx_search(control_query, language=language, k=10)
        urls = {str(item.get("url") or "") for item in results if item.get("url")}
        controls[label] = urls
        tlds, hosts = result_profile(results)
        expected = control_targets.get(label)
        expected_hits = sum(expected in urlparse(url).netloc.lower() for url in urls) if expected else 0
        overlap = len(urls & controls["none"]) / len(urls | controls["none"]) if label != "none" and (urls | controls["none"]) else 1.0
        print(f"control language={language or '<none>'}: TLDs={dict(tlds)}, expected hits={expected_hits}, Jaccard vs none={overlap:.2f}")

    language_only_success = all(
        any(target in urlparse(url).netloc.lower() for url in controls[country])
        for country, target in control_targets.items()
    )
    retailer_keyword_success = any(local_target_hits.values())
    print("\n结论：")
    if language_only_success:
        print("能定向（本次样本中 language-only 对照覆盖了所有预期国家域名；扩大样本后再确认）。")
    elif retailer_keyword_success:
        print("只能靠 query 里的本地零售商/国家关键词兜底；language 会改变结果，但不能等同于国家参数。")
    else:
        print("完全不能可靠定向；language 和 query 关键词均未带回目标国家域名。")


uk / en-GB: 0 results, target hits=0
    TLDs: {}
    domains: []
de / de-DE: 0 results, target hits=0
    TLDs: {}
    domains: []
fr / fr-FR: 0 results, target hits=0
    TLDs: {}
    domains: []
us / en-US: 0 results, target hits=0
    TLDs: {}
    domains: []
control language=<none>: TLDs={}, expected hits=0, Jaccard vs none=1.00
control language=en-GB: TLDs={}, expected hits=0, Jaccard vs none=1.00
control language=de-DE: TLDs={}, expected hits=0, Jaccard vs none=1.00
control language=fr-FR: TLDs={}, expected hits=0, Jaccard vs none=1.00
control language=en-US: TLDs={}, expected hits=0, Jaccard vs none=1.00

结论：
完全不能可靠定向；language 和 query 关键词均未带回目标国家域名。


In [7]:
# Cell 6: local and upstream rate-limit behavior
if not sx_up():
    print(sx_unavailable_message())
else:
    QUERY = "Nescafe Gold Instant Coffee 200g Tesco"
    N_CALLS = 15

    def test_rate(delay_s: float, n: int = N_CALLS) -> dict:
        """Run n queries with a fixed delay and return DDG-compatible stats plus engine health."""
        times: list[float] = []
        errors = 0
        result_counts: list[int] = []
        unresponsive_counts: Counter = Counter()
        for index in range(n):
            if delay_s > 0:
                time.sleep(delay_s)
            started = time.perf_counter()
            try:
                results, meta = sx_search(QUERY, k=5)
                elapsed = meta["elapsed_s"]
                result_counts.append(len(results))
                for engine_state in meta["unresponsive_engines"]:
                    engine = engine_state[0] if isinstance(engine_state, (list, tuple)) and engine_state else str(engine_state)
                    unresponsive_counts[str(engine)] += 1
            except Exception as exc:
                elapsed = time.perf_counter() - started
                result_counts.append(0)
                errors += 1
                print(f"  call {index + 1} ERROR ({elapsed:.2f}s): {type(exc).__name__}: {str(exc)[:100]}")
            times.append(elapsed)
            if index < 3 or index == n - 1:
                print(f"  call {index + 1}: {result_counts[-1]} results in {elapsed:.2f}s")
        return {
            "delay": delay_s,
            "errors": errors,
            "avg_time": sum(times) / len(times) if times else 0,
            "min_time": min(times) if times else 0,
            "max_time": max(times) if times else 0,
            "avg_results": sum(result_counts) / len(result_counts) if result_counts else 0,
            "empty_calls": sum(count == 0 for count in result_counts),
            "unresponsive_engines": dict(unresponsive_counts),
        }

    def run_rate_test() -> list[dict]:
        """Run the destructive stress test after all result-quality cells."""
        rate_stats: list[dict] = []
        for delay in (0.0, 1.0, 2.0):
            print("=" * 60)
            print(f"Testing delay={delay:.0f}s ({N_CALLS} calls)")
            rate_stats.append(test_rate(delay, n=N_CALLS))

        print("\nSUMMARY")
        for stats in rate_stats:
            print(
                f"delay={stats['delay']:.0f}s: {stats['errors']} HTTP errors, "
                f"{stats['empty_calls']} empty calls, "
                f"{stats['avg_time']:.2f}s avg ({stats['min_time']:.2f}-{stats['max_time']:.2f}), "
                f"{stats['avg_results']:.1f} avg results, "
                f"unresponsive={stats['unresponsive_engines']}"
            )
        return rate_stats

    print("Rate-test helper ready; the 45-call stress test is deferred until after cell 8 so it cannot poison quality checks.")


Rate-test helper ready; the 45-call stress test is deferred until after cell 8 so it cannot poison quality checks.


In [8]:
# Cell 7: the same five edge cases as duckduckgo.ipynb
if not sx_up():
    print(sx_unavailable_message())
else:
    edge_cases = [
        ("AAA", "very short query"),
        ("Kühlschrank mit Gefrierfach Edelstahl 180cm Energieklasse A+++ Siemens", "long German query with umlauts"),
        ("Crème Brûlée à la vanille de Madagascar 500ml", "French accents"),
        ("xyznonexistentproduct12345", "nonsense — likely no results"),
        ("Sony WH-1000XM5 Wireless Noise Cancelling Headphones Black", "very common product — many results"),
    ]
    for query, description in edge_cases:
        try:
            results, meta = sx_search(query, k=5)
        except Exception as exc:
            print(f"[{description}] ERROR: {type(exc).__name__}: {str(exc)[:150]}")
            continue
        bad_titles = sum(not str(item.get("title") or "").strip() for item in results)
        bad_urls = sum(not str(item.get("url") or "").strip() for item in results)
        print(f"[{description}]")
        print(f"    Query: {query[:80]!r}{'...' if len(query) > 80 else ''}")
        print(f"    {len(results)} results, {bad_titles} empty titles, {bad_urls} empty URLs, {meta['elapsed_s']:.2f}s")
        if results:
            print(f"    First: {str(results[0].get('title') or 'N/A')[:100]}")


[very short query]
    Query: 'AAA'
    1 results, 0 empty titles, 0 empty URLs, 0.06s
    First: AAA
[long German query with umlauts]
    Query: 'Kühlschrank mit Gefrierfach Edelstahl 180cm Energieklasse A+++ Siemens'
    0 results, 0 empty titles, 0 empty URLs, 0.26s
[French accents]
    Query: 'Crème Brûlée à la vanille de Madagascar 500ml'
    0 results, 0 empty titles, 0 empty URLs, 0.86s
[nonsense — likely no results]
    Query: 'xyznonexistentproduct12345'
    0 results, 0 empty titles, 0 empty URLs, 0.36s
[very common product — many results]
    Query: 'Sony WH-1000XM5 Wireless Noise Cancelling Headphones Black'
    0 results, 0 empty titles, 0 empty URLs, 0.43s


In [9]:
# Cell 8: direct SearXNG vs DDGS comparison
if not sx_up():
    print(sx_unavailable_message())
else:
    try:
        from ddgs import DDGS
    except ImportError:
        print("ddgs is unavailable in this environment; comparison skipped (no package is installed by this notebook).")
    else:
        comparison_queries = [
            ("Heinz Baked Beans 415g Tesco", "tesco.com"),
            ("Samsung Galaxy S24 Ultra Amazon", "amazon."),
            ("Bosch Serie 4 Washing Machine Argos", "argos.co.uk"),
            ("Kopparberg Strawberry & Lime 500ml", None),
            ("LEGO Star Wars Millennium Falcon", None),
        ]

        def normalized_url(value: str) -> str:
            return value.split("#", 1)[0].rstrip("/")

        sx_times: list[float] = []
        ddg_times: list[float] = []
        ddgs = DDGS()
        for query, target_domain in comparison_queries:
            sx_results, sx_meta = sx_search(query, language="en-GB", k=10)
            ddg_started = time.perf_counter()
            try:
                ddg_results = list(ddgs.text(query, max_results=10, region="uk-en"))
            except Exception as exc:
                ddg_results = []
                print(f"DDG ERROR for {query!r}: {type(exc).__name__}: {str(exc)[:120]}")
            ddg_elapsed = time.perf_counter() - ddg_started
            sx_times.append(sx_meta["elapsed_s"])
            ddg_times.append(ddg_elapsed)
            sx_urls = {normalized_url(str(item.get("url") or "")) for item in sx_results if item.get("url")}
            ddg_urls = {normalized_url(str(item.get("href") or "")) for item in ddg_results if item.get("href")}
            union = sx_urls | ddg_urls
            overlap = len(sx_urls & ddg_urls) / len(union) if union else 1.0
            sx_target_hits = sum(target_domain in urlparse(url).netloc.lower() for url in sx_urls) if target_domain else 0
            ddg_target_hits = sum(target_domain in urlparse(url).netloc.lower() for url in ddg_urls) if target_domain else 0
            print(f"\n{query}")
            print(f"  SearXNG={len(sx_urls)}, DDG={len(ddg_urls)}, URL Jaccard={overlap:.2f}")
            print(f"  target hits: SearXNG={sx_target_hits}, DDG={ddg_target_hits}")
            print(f"  SearXNG-only: {sorted(sx_urls - ddg_urls)}")
            print(f"  DDG-only: {sorted(ddg_urls - sx_urls)}")

        print("\nAVERAGE LATENCY")
        print(f"  SearXNG: {sum(sx_times) / len(sx_times):.2f}s")
        print(f"  DDG:     {sum(ddg_times) / len(ddg_times):.2f}s")

if sx_up() and "run_rate_test" in globals():
    print("\n--- Deferred rate-limit test (runs last) ---")
    rate_stats = run_rate_test()
elif "run_rate_test" not in globals():
    print("Rate test not defined; run cell 6 first.")



Heinz Baked Beans 415g Tesco
  SearXNG=0, DDG=10, URL Jaccard=0.00
  target hits: SearXNG=0, DDG=5
  SearXNG-only: []
  DDG-only: ['https://origin-main.mfe.ppe.omnichannel.tescocloud.com/groceries/en-GB/products/273724002', 'https://www.glasgowtimes.co.uk/news/national/uk-today/23640766.heinz-beans-prices-tesco-sainsburys-asda', 'https://www.tesco.com/groceries/en-GB/products/250741978', 'https://www.tesco.com/groceries/en-GB/products/252004443', 'https://www.tesco.com/groceries/en-GB/products/253034146', 'https://www.tesco.com/shop/en-GB/products/252261477', 'https://www.tesco.com/shop/en-GB/products/321339973', 'https://www.tesco.ie/shop/en-IE/products/273724002', 'https://www.tesco.ie/shop/en-IE/products/295444806', 'https://www.trolley.co.uk/product/heinz-beanz-family-pack/DJC713']

Samsung Galaxy S24 Ultra Amazon
  SearXNG=0, DDG=10, URL Jaccard=0.00
  target hits: SearXNG=0, DDG=10
  SearXNG-only: []
  DDG-only: ['https://www.amazon.com/SAMSUNG-Galaxy-Ultra-Version-Titanium/dp/B

# Cell 9: Summary & Recommendation

Recorded after the 2026-08-14 end-to-end run.

## Public-instance baseline

On 2026-08-14, 0/17 tested public instances exposed a usable `format=json` endpoint. Seven returned HTTP 429, four returned anti-bot/captcha HTML with HTTP 200, and six were forbidden, unavailable, unreachable, or retired. The official API documentation also notes that JSON is disabled by default and many public instances leave it disabled. **This feasibility result therefore applies to a self-hosted instance, not public-instance availability.**

## Field contract and result quality

- `url / title / content` mapped cleanly to `RawCandidate.url / title / snippet`; no transformation errors were observed.
- Before upstream throttling, the basic query returned 10 candidates including 3 Tesco URLs.
- All five format samples had zero missing URL/title fields.
- Before throttling, SearXNG returned many marketplace URLs absent from DDG and averaged about 0.48s versus DDG's 1.24s on the five-query comparison.

## Country targeting

- `language=` materially changed result sets and TLDs, but it is not a country-location parameter.
- UK, German, and French samples produced their expected local domains; the US sample did not reliably produce Amazon US.
- Retailer/country terms are still required in the query.
- Conclusion: query-keyword fallback only; this does not satisfy reliable country targeting.

## Rate limits and upstream health

- The first 15 no-delay calls returned 5 results each, averaging about 0.40s.
- During the next 15 calls at 1s delay, results degraded to an average of 3.3 and then became empty.
- The final 15 calls at 2s delay all returned empty results despite HTTP 200 responses.
- No safe sustained call rate was demonstrated; a slower delay did not recover already-suspended upstreams.
- Blocked/degraded upstreams: Brave (429), DuckDuckGo (CAPTCHA), Startpage (CAPTCHA), and eventually Google CSE.

## Edge cases

- Short queries were handled without malformed rows.
- German umlauts and French accents returned valid results.
- A nonsense query returned a clean empty list.

## Overall verdict

- [ ] **Viable** — country targeting, stability, and quality all meet the provider contract
- [ ] **Conditionally viable** — useful only with explicit constraints documented below
- [x] **Not viable** — do not implement a production provider with the tested default upstreams/network egress

### If implementing a provider later

- Validate and finalize `_COUNTRY_TO_LANGUAGE` for all 15 accepted country codes.
- Decide whether `engines=` must pin a known-good upstream set.
- Specify deployment, monitoring, upgrades, and operational ownership for the self-hosted instance.
- Rewrite the synchronous spike with the production provider's `aiohttp` async contract.
- Define retry/backoff from the upstream-engine failures observed above.
